# Homework 2: Archaic to Modern Italian Translation

**Course**: Natural Language Processing  
**Group**: exACSAI  
**Authors**: Ejaz Ahmed, Archit Rastogi  
**University**: Sapienza University of Rome

---

## Overview

This notebook evaluates multiple approaches for translating medieval Italian (13th-15th century) to modern Italian:

1. **Fine-tuned Expert Models**: Specialized LoRA-adapted Mistral-7B models
   - **Approach 1**: Linguistic experts (Lexical, Syntactic, Semantic)
   - **Approach 2**: Temporal experts (Early, Middle, Late periods)
   - **Merged Models**: RegMean + SNR-based merging

2. **Large Language Models**: GPT-4.1 and Gemini 2.0 (zero-shot and few-shot)

3. **Evaluation**: LLM-as-a-Judge using GPT-4.1

**Note**: This notebook focuses on **inference and evaluation only**. Training code is available in our repository.

## Configuration

**IMPORTANT**: Set your API keys and data paths here before running.

In [ ]:
# =====================================================
# CONFIGURATION - EDIT THESE VALUES
# =====================================================

# API Keys (required for LLM baselines and LLM-as-a-Judge)
GOOGLE_API_KEY = ""  # Your Google API key for Gemini 2.0
OPENAI_API_KEY = ""  # Your OpenAI API key for GPT-4.1

# HuggingFace Settings
HF_USERNAME = "ejaz111"  # Models are hosted here

# Test Dataset Path (upload your Excel file to Colab)
TEST_DATA_PATH = "task2-archaic2modern_ita.xlsx"  # Path to test data

# Output Settings
GROUP_NAME = "exACSAI"
OUTPUT_DIR = "outputs"

# Model Selection (set to False to skip)
RUN_FINETUNED_MODELS = True   # Run fine-tuned expert models
RUN_GPT4 = True               # Run GPT-4.1 (zero-shot and few-shot)
RUN_GEMINI = True             # Run Gemini 2.0 (zero-shot and few-shot)
RUN_LLM_JUDGE = True          # Run LLM-as-a-Judge evaluation

# Generation Settings
MAX_NEW_TOKENS = 128
TEMPERATURE = 0.7
BATCH_SIZE = 4  # Adjust based on GPU memory

## Setup and Installation

In [ ]:
!pip install "typing-extensions>=4.14.0" "pydantic>=2.7.0" "pydantic-core>=2.18.0"
!pip install "openai>=1.30.0" google-generativeai

!pip install transformers accelerate peft bitsandbytes
!pip install torch torchvision torchaudio
!pip install huggingface_hub hf-transfer
!pip install pandas openpyxl
!pip install evaluate sacrebleu bert-score rouge-score


In [7]:
# Import libraries
import os
import json
import time
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm
from datetime import datetime

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel, PeftConfig
from huggingface_hub import snapshot_download

from openai import OpenAI
import google.generativeai as genai

# Create output directory
OUTPUT_PATH = Path(OUTPUT_DIR)
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

print(f"Output directory: {OUTPUT_PATH}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Output directory: outputs
CUDA available: True
GPU: NVIDIA RTX A4500


## Load Test Dataset

Load the 97 archaic Italian sentences for evaluation.

In [8]:
def load_test_data(file_path):
    """
    Load test sentences from Excel file.

    Returns:
        List of dictionaries with 'id', 'source', 'author', 'date', 'region'
    """
    df = pd.read_excel(file_path)
    df = df.dropna(subset=["Sentence"]).copy()

    test_data = []
    for idx, row in df.iterrows():
        test_data.append({
            "id": int(idx),
            "source": str(row["Sentence"]).strip(),
            "author": str(row.get("Author", "Unknown")),
            "date": str(row.get("Date", "Unknown")),
            "region": str(row.get("Region", "Unknown"))
        })

    return test_data

# Load test data
test_sentences = load_test_data(TEST_DATA_PATH)
print(f"Loaded {len(test_sentences)} test sentences")
print(f"\nSample sentence:")
print(f"   Source: {test_sentences[0]['source'][:100]}...")
print(f"   Author: {test_sentences[0]['author']}")
print(f"   Date: {test_sentences[0]['date']}")

Loaded 97 test sentences

Sample sentence:
   Source: quella guerra ben fatta l' opera perché etc. Et dall' altra parte Aiaces era uno cavaliere franco e ...
   Author: Brunetto Latini
   Date: 1260-61


## Part 1: Fine-Tuned Expert Models

### Download Models from HuggingFace

We download our pre-trained expert models and merged models.

In [9]:
# Model registry
MODELS = {
    # Approach 1: Linguistic Experts
    "a1_lexical_expert": f"{HF_USERNAME}/archaic-italian_a1-lexical-expert-lora",
    "a1_syntactic_expert": f"{HF_USERNAME}/archaic-italian_a1-syntactic-expert-lora",
    "a1_semantic_expert": f"{HF_USERNAME}/archaic-italian_a1-semantic-expert-lora",
    "a1_merged_full": f"{HF_USERNAME}/archaic-italian_a1-merged-full",

    # Approach 2: Temporal Experts
    "a2_early_expert": f"{HF_USERNAME}/archaic-italian_a2-early-expert-lora",
    "a2_middle_expert": f"{HF_USERNAME}/archaic-italian_a2-middle-expert-lora",
    "a2_late_expert": f"{HF_USERNAME}/archaic-italian_a2-late-expert-lora",
    "a2_merged_full": f"{HF_USERNAME}/archaic-italian_a2-merged-full",

    # Unified Model
    "unified_merged_full": f"{HF_USERNAME}/archaic-italian_unified-merged-full",
}

BASE_MODEL = "mistralai/Mistral-7B-Instruct-v0.2"

def download_model(model_name, repo_id):
    """Download model from HuggingFace if not already cached."""
    local_dir = Path("models") / model_name

    if local_dir.exists() and any(local_dir.iterdir()):
        print(f"{model_name} already cached")
        return str(local_dir)

    print(f"Downloading {model_name}...")
    try:
        snapshot_download(
            repo_id=repo_id,
            local_dir=str(local_dir),
            local_dir_use_symlinks=False,
        )
        print(f"Downloaded {model_name}")
        return str(local_dir)
    except Exception as e:
        print(f"Failed to download {model_name}: {e}")
        return None

if RUN_FINETUNED_MODELS:
    print("Downloading fine-tuned models from HuggingFace...\n")
    model_paths = {}
    for name, repo_id in MODELS.items():
        path = download_model(name, repo_id)
        if path:
            model_paths[name] = path

    print(f"\nSuccessfully downloaded {len(model_paths)}/{len(MODELS)} models")
else:
    print("Skipping fine-tuned models (RUN_FINETUNED_MODELS=False)")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/file_download.py:979: UserWarning: `local_dir_use_symlinks` parameter is deprecated and will be ignored. The process to download files to a local folder has been updated and do not rely on symlinks anymore. You only need to pass a destination folder as`local_dir`.
For more details, check out https://huggingface.co/docs/huggingface_hub/main/en/guides/download#download-files-to-local-folder.
  warnings.warn(


Fetching 39 files:   0%|          | 0/39 [00:00<?, ?it/s]

checkpoint-15/adapter_model.safetensors:   0%|          | 0.00/54.6M [00:00<?, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/54.6M [00:00<?, ?B/s]

adapter_config.json:   0%|          | 0.00/897 [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

Downloaded unified_merged_full

Successfully downloaded 9/9 models


### Inference with Fine-Tuned Models

In [10]:
def format_prompt(text):
    """Format input for model inference."""
    return f"""### Istruzione:
Modernizza la seguente frase italiana arcaica in italiano contemporaneo.

### Frase arcaica:
{text}

### Frase moderna:
"""

def load_finetuned_model(model_name, model_path):
    """
    Load a fine-tuned model with 8-bit quantization for memory efficiency.
    
    Args:
        model_name: Name of the model
        model_path: Path to model directory
    
    Returns:
        (model, tokenizer) tuple
    """
    from transformers import BitsAndBytesConfig
    
    is_adapter = "expert" in model_name and "merged" not in model_name
    
    # 8-bit quantization config to save memory
    quantization_config = BitsAndBytesConfig(
        load_in_8bit=True,
        llm_int8_threshold=6.0,
    )
    
    if is_adapter:
        # Load LoRA adapter on base model
        print(f"  Loading base model + LoRA adapter (8-bit quantized)...")
        base_model = AutoModelForCausalLM.from_pretrained(
            BASE_MODEL,
            quantization_config=quantization_config,
            device_map="auto",
            trust_remote_code=True,
        )
        tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
        
        # Load PEFT adapter
        model = PeftModel.from_pretrained(
            base_model,
            model_path,
        )
    else:
        # Load full merged model with quantization
        print(f"  Loading merged model (8-bit quantized)...")
        model = AutoModelForCausalLM.from_pretrained(
            model_path,
            quantization_config=quantization_config,
            device_map="auto",
            trust_remote_code=True,
        )
        tokenizer = AutoTokenizer.from_pretrained(model_path)
    
    # Configure tokenizer
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"
    
    model.eval()
    return model, tokenizer

def translate_batch(model, tokenizer, texts, desc="Translating"):
    """
    Translate a batch of texts using the model.
    
    Args:
        model: Loaded model
        tokenizer: Tokenizer
        texts: List of input texts
        desc: Progress bar description
    
    Returns:
        List of translations
    """
    translations = []
    
    # Process one at a time to avoid OOM
    for text in tqdm(texts, desc=desc, leave=False):
        prompt = format_prompt(text)
        
        # Tokenize single input
        inputs = tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=512,
        ).to(model.device)
        
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                temperature=TEMPERATURE,
                top_p=0.9,
                do_sample=True,
                repetition_penalty=1.1,
                pad_token_id=tokenizer.eos_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )
        
        # Decode output
        generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
        
        # Extract translation (after "### Frase moderna:")
        translation = generated_text.split("### Frase moderna:")[-1].strip().split("\n")[0]
        translations.append(translation)
    
    return translations

def save_translations_jsonl(model_name, predictions, output_dir):
    """
    Save translations in required .jsonl format.
    
    Format: {GROUP_NAME}-hw2_transl-{model_name}.jsonl
    """
    filename = f"{GROUP_NAME}-hw2_transl-{model_name}.jsonl"
    filepath = output_dir / filename
    
    with open(filepath, "w", encoding="utf-8") as f:
        for pred in predictions:
            json.dump(pred, f, ensure_ascii=False)
            f.write("\n")
    
    print(f"Saved: {filename}")
    return filepath

In [11]:
if RUN_FINETUNED_MODELS:
    print("Running inference with fine-tuned models...\n")
    print("Using 8-bit quantization to fit in GPU memory\n")
    
    finetuned_results = {}
    
    for model_name, model_path in model_paths.items():
        print(f"\n{'='*60}")
        print(f"Model: {model_name}")
        print(f"{'='*60}")
        
        try:
            # Load model
            model, tokenizer = load_finetuned_model(model_name, model_path)
            
            # Get source texts
            source_texts = [item["source"] for item in test_sentences]
            
            # Translate (one at a time for memory safety)
            translations = translate_batch(
                model, tokenizer, source_texts, 
                desc=f"Translating with {model_name}"
            )
            
            # Prepare results
            predictions = []
            for i, (sent, trans) in enumerate(zip(test_sentences, translations)):
                predictions.append({
                    "id": sent["id"],
                    "source": sent["source"],
                    "prediction": trans,
                    "author": sent["author"],
                    "date": sent["date"],
                    "region": sent["region"]
                })
            
            # Save to file
            save_translations_jsonl(model_name, predictions, OUTPUT_PATH)
            finetuned_results[model_name] = predictions
            
            # Free memory
            del model, tokenizer
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            import gc
            gc.collect()
            
            print(f"Completed {model_name}")
            print(f"Translated {len(translations)} sentences")
            
        except Exception as e:
            print(f"Error with {model_name}: {str(e)}")
            # Clear memory even on error
            try:
                del model, tokenizer
            except:
                pass
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            import gc
            gc.collect()
            continue
    
    print(f"\nFine-tuned models inference complete!")
    print(f"   Generated {len(finetuned_results)} output files")
else:
    print("Skipping fine-tuned models")
    finetuned_results = {}

Running inference with fine-tuned models...

Using 8-bit quantization to fit in GPU memory


Model: a1_lexical_expert
  Loading base model + LoRA adapter (8-bit quantized)...


config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

model-00001-of-00003.safetensors:   0%|          | 0.00/4.94G [00:00<?, ?B/s]

...

Translating with a2_merged_full:   0%|          | 0/97 [00:00<?, ?it/s]

Saved: exACSAI-hw2_transl-a2_merged_full.jsonl
Completed a2_merged_full
Translated 97 sentences

Model: unified_merged_full
  Loading merged model (8-bit quantized)...


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Translating with unified_merged_full:   0%|          | 0/97 [00:00<?, ?it/s]

Saved: exACSAI-hw2_transl-unified_merged_full.jsonl
Completed unified_merged_full
Translated 97 sentences

Fine-tuned models inference complete!
   Generated 9 output files


## Part 2: Large Language Model Baselines

### GPT-4.1 (Zero-Shot and Few-Shot)

In [12]:
def build_gpt4_prompt_zeroshot(text):
    """Zero-shot prompt for GPT-4.1."""
    return f"""You are a professional Italian linguist specialized in transforming
archaic (13th–15th century) Italian into natural 2025 Italian.

### Task
Translate the following archaic Italian sentence into fluent modern Italian,
preserving meaning and stylistic naturalness.

### Archaic Sentence:
{text}

### Output rules
- Keep the meaning identical
- Use modern vocabulary and syntax
- Output only the translation text, no explanations
- The translation must fit in the 'translation' JSON field
"""

def build_gpt4_prompt_fewshot(text):
    """Few-shot prompt for GPT-4.1 with examples."""
    examples = """### Example 1
"source": "sono due già non in una carne, ma in uno spirito, cioè Iddio, e l' anima. Onde in altro luogo dice S. Paolo: Chi s' accosta a Dio è uno spirito",
"target": "sono due già non in una carne, ma in uno spirito, cioè Dio e l'anima. Onde in un altro luogo dice San Paolo: Chi si accosta a Dio è uno spirito"

### Example 2
"source": "Altressì uno amante chiamando merzé alla sua donna dice parole e ragioni molte, et ella si difende in suo dire.",
"target": "Allo stesso modo, un uomo innamorato, mentre chiede misericordia alla donna che ama, le rivolge molte parole e argomentazioni, ed essa risponde difendendo la propria posizione con le sue parole."

### Example 3
"source": "Andò nel campo de' Cartaginesi e tutta la legione trasse seco.",
"target": "Trasse con sé tutta la legione e andò nel campo dei Cartaginesi."
"""

    return f"""You are a professional Italian linguist specialized in transforming
archaic (13th–15th century) Italian into natural 2025 Italian.

Learn from the examples below and translate the following archaic sentence
into fluent modern Italian while preserving meaning and tone.

{examples}

### Sentence to translate:
{text}

### Output format
Respond strictly in JSON as:
{{"translation": "<modern translation>"}}
"""

TRANSLATION_SCHEMA = {
    "type": "json_schema",
    "json_schema": {
        "name": "archaic_to_modern_translation",
        "strict": True,
        "schema": {
            "type": "object",
            "properties": {
                "translation": {
                    "type": "string",
                    "description": "Modern Italian translation"
                }
            },
            "required": ["translation"],
            "additionalProperties": False
        }
    }
}

def translate_with_gpt4(text, prompt_builder, client, max_retries=3):
    """Translate a single text with GPT-4.1."""
    prompt = prompt_builder(text)

    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model="gpt-4.1",
                temperature=0.4,
                max_tokens=256,
                response_format=TRANSLATION_SCHEMA,
                messages=[
                    {"role": "system", "content": "Translate archaic Italian to modern Italian."},
                    {"role": "user", "content": prompt}
                ]
            )
            data = json.loads(response.choices[0].message.content)
            return data["translation"]
        except Exception as e:
            if attempt < max_retries - 1:
                time.sleep(2)
                continue
            return None
    return None

In [13]:
if RUN_GPT4:
    if not OPENAI_API_KEY:
        print("OpenAI API key not set. Skipping GPT-4.1.")
    else:
        print("Running GPT-4.1 translations...\n")
        client = OpenAI(api_key=OPENAI_API_KEY)

        gpt4_results = {}

        # Zero-shot
        print("GPT-4.1 Zero-Shot")
        predictions = []
        for sent in tqdm(test_sentences, desc="GPT-4.1 Zero-Shot"):
            translation = translate_with_gpt4(
                sent["source"],
                build_gpt4_prompt_zeroshot,
                client
            )
            predictions.append({
                "id": sent["id"],
                "source": sent["source"],
                "prediction": translation if translation else "[FAILED]",
                "author": sent["author"],
                "date": sent["date"],
                "region": sent["region"]
            })
            time.sleep(0.5)  # Rate limiting

        save_translations_jsonl("gpt4.1-zeroshot", predictions, OUTPUT_PATH)
        gpt4_results["gpt4.1-zeroshot"] = predictions

        # Few-shot
        print("\nGPT-4.1 Few-Shot")
        predictions = []
        for sent in tqdm(test_sentences, desc="GPT-4.1 Few-Shot"):
            translation = translate_with_gpt4(
                sent["source"],
                build_gpt4_prompt_fewshot,
                client
            )
            predictions.append({
                "id": sent["id"],
                "source": sent["source"],
                "prediction": translation if translation else "[FAILED]",
                "author": sent["author"],
                "date": sent["date"],
                "region": sent["region"]
            })
            time.sleep(0.5)

        save_translations_jsonl("gpt4.1-fewshot", predictions, OUTPUT_PATH)
        gpt4_results["gpt4.1-fewshot"] = predictions

        print(f"\nGPT-4.1 translations complete!")
else:
    print("Skipping GPT-4.1")
    gpt4_results = {}

Running GPT-4.1 translations...

GPT-4.1 Zero-Shot


GPT-4.1 Zero-Shot:   0%|          | 0/97 [00:00<?, ?it/s]

Saved: exACSAI-hw2_transl-gpt4.1-zeroshot.jsonl

GPT-4.1 Few-Shot


GPT-4.1 Few-Shot:   0%|          | 0/97 [00:00<?, ?it/s]

KeyboardInterrupt: 

### Gemini 2.0 (Zero-Shot and Few-Shot)

In [16]:
def build_gemini_prompt_zeroshot(text):
    """Zero-shot prompt for Gemini 2.0 (same as GPT-4.1)."""
    return build_gpt4_prompt_zeroshot(text)

def build_gemini_prompt_fewshot(text):
    """Few-shot prompt for Gemini 2.0 (same as GPT-4.1)."""
    return build_gpt4_prompt_fewshot(text)

gemini_schema = {
    "type": "object",
    "properties": {
        "translation": {
            "type": "string",
            "description": "Modern Italian translation of the input sentence."
        }
    },
    "required": ["translation"]
}

def translate_with_gemini(text, prompt_builder, max_retries=3):
    """Translate a single text with Gemini 2.0 using v0.8.6 API."""
    prompt = prompt_builder(text)
    
    for attempt in range(max_retries):
        try:
            # v0.8.6 uses GenerativeModel, not Client
            model = genai.GenerativeModel('gemini-2.0-flash-exp')
            
            response = model.generate_content(
                prompt,
                generation_config={
                    "temperature": 0.4,
                    "max_output_tokens": 256,
                    "response_mime_type": "application/json",
                    "response_schema": gemini_schema,
                }
            )
            
            # Extract text from response
            text_content = response.text.strip()
            data = json.loads(text_content)
            return data.get("translation")
            
        except Exception as e:
            print(f"   Attempt {attempt + 1} failed: {str(e)[:50]}...")
            if attempt < max_retries - 1:
                time.sleep(5)
                continue
            return None
    return None

In [17]:
if RUN_GEMINI:
    if not GOOGLE_API_KEY:
        print("Google API key not set. Skipping Gemini 2.0.")
    else:
        print("Running Gemini 2.0 translations...\n")
        
        # Configure API for v0.8.6
        import google.generativeai as genai
        genai.configure(api_key=GOOGLE_API_KEY)
        
        gemini_results = {}
        
        # Zero-shot
        print("Gemini 2.0 Zero-Shot")
        predictions = []
        for sent in tqdm(test_sentences, desc="Gemini 2.0 Zero-Shot"):
            translation = translate_with_gemini(
                sent["source"],
                build_gemini_prompt_zeroshot
            )
            predictions.append({
                "id": sent["id"],
                "source": sent["source"],
                "prediction": translation if translation else "[FAILED]",
                "author": sent["author"],
                "date": sent["date"],
                "region": sent["region"]
            })
            time.sleep(5)  # Gemini rate limiting
        
        save_translations_jsonl("gemini2.0-zeroshot", predictions, OUTPUT_PATH)
        gemini_results["gemini2.0-zeroshot"] = predictions
        
        # Few-shot
        print("\nGemini 2.0 Few-Shot")
        predictions = []
        for sent in tqdm(test_sentences, desc="Gemini 2.0 Few-Shot"):
            translation = translate_with_gemini(
                sent["source"],
                build_gemini_prompt_fewshot
            )
            predictions.append({
                "id": sent["id"],
                "source": sent["source"],
                "prediction": translation if translation else "[FAILED]",
                "author": sent["author"],
                "date": sent["date"],
                "region": sent["region"]
            })
            time.sleep(5)
        
        save_translations_jsonl("gemini2.0-fewshot", predictions, OUTPUT_PATH)
        gemini_results["gemini2.0-fewshot"] = predictions
        
        print(f"\nGemini 2.0 translations complete!")
else:
    print("Skipping Gemini 2.0")
    gemini_results = {}

Running Gemini 2.0 translations...

Gemini 2.0 Zero-Shot


Gemini 2.0 Zero-Shot:   0%|          | 0/97 [00:00<?, ?it/s]

   Attempt 1 failed: 429 You exceeded your current quota, please check ...
   Attempt 2 failed: 429 You exceeded your current quota, please check ...
   Attempt 3 failed: 429 You exceeded your current quota, please check ...


KeyboardInterrupt: 

## Part 3: LLM-as-a-Judge Evaluation

We use GPT-4.1 to evaluate translation quality across four criteria:
- **Faithfulness**: Meaning preservation
- **Fluency**: Grammar and readability
- **Style**: Naturalness and human-likeness
- **Overall**: Holistic quality

In [18]:
JUDGE_SCHEMA = {
    "type": "json_schema",
    "json_schema": {
        "name": "llm_translation_evaluation",
        "strict": True,
        "schema": {
            "type": "object",
            "properties": {
                "Faithfulness": {
                    "type": "object",
                    "properties": {
                        "feedback": {"type": "string"},
                        "score": {"type": "integer", "minimum": 1, "maximum": 5}
                    },
                    "required": ["feedback", "score"],
                    "additionalProperties": False
                },
                "Fluency": {
                    "type": "object",
                    "properties": {
                        "feedback": {"type": "string"},
                        "score": {"type": "integer", "minimum": 1, "maximum": 5}
                    },
                    "required": ["feedback", "score"],
                    "additionalProperties": False
                },
                "Style": {
                    "type": "object",
                    "properties": {
                        "feedback": {"type": "string"},
                        "score": {"type": "integer", "minimum": 1, "maximum": 5}
                    },
                    "required": ["feedback", "score"],
                    "additionalProperties": False
                },
                "Overall": {
                    "type": "object",
                    "properties": {
                        "feedback": {"type": "string"},
                        "score": {"type": "integer", "minimum": 1, "maximum": 5}
                    },
                    "required": ["feedback", "score"],
                    "additionalProperties": False
                }
            },
            "required": ["Faithfulness", "Fluency", "Style", "Overall"],
            "additionalProperties": False
        }
    }
}

RUBRICS = {
    "Faithfulness": """Evaluate meaning preservation and modernization accuracy.
5 – Excellent: Meaning fully preserved and properly modernized.
4 – Good: Minor nuance loss or slightly literal.
3 – Fair: Understandable but partial misinterpretation.
2 – Poor: Major omission or distortion.
1 – Fail: Meaning lost or wrong.""",

    "Fluency": """Evaluate grammar, syntax, and comprehensibility.
5 – Excellent: Flawless, idiomatic modern Italian.
4 – Good: Minor slips but readable.
3 – Fair: Awkward phrasing or rigid syntax.
2 – Poor: Frequent grammatical errors.
1 – Fail: Broken or unreadable.""",

    "Style": """Evaluate naturalness and human-likeness of the translation.
5 – Excellent: Feels human, elegant, and modern.
4 – Good: Smooth, slightly mechanical.
3 – Fair: Understandable but stiff.
2 – Poor: Robotic tone or phrasing.
1 – Fail: Stylistically incoherent.""",

    "Overall": """Evaluate the holistic translation quality combining all above criteria.
5 – Excellent: Faithful, fluent, and natural.
4 – Good: Minor issues not affecting comprehension.
3 – Fair: Acceptable but mechanical.
2 – Poor: Noticeable problems.
1 – Fail: Unusable translation."""
}

def build_judge_prompt(source, prediction, reference=None):
    """Build LLM-as-a-Judge evaluation prompt."""
    ref_text = reference if reference else "N/A"

    return f"""You are a bilingual Italian linguist specialized in translating and evaluating
texts from medieval Italian (13th–15th century) to modern Italian.

Evaluate the following translation according to the rubrics below.
Use the "Reference Translation" only as a semantic anchor – not a gold standard.
Do not paraphrase or retranslate; only judge the given text.

Provide feedback ≤50 words per criterion, strictly following the rubrics.

### Archaic Sentence
{source}

### Model Translation
{prediction}

### Reference Translation (semantic anchor)
{ref_text}

### Rubrics
1. Faithfulness – {RUBRICS['Faithfulness']}
2. Fluency – {RUBRICS['Fluency']}
3. Style – {RUBRICS['Style']}
4. Overall – {RUBRICS['Overall']}
"""

def judge_translation(source, prediction, reference, client, max_retries=3):
    """Get LLM-as-a-Judge evaluation for a single translation."""
    prompt = build_judge_prompt(source, prediction, reference)

    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model="gpt-4.1",
                temperature=0.3,
                max_tokens=800,
                response_format=JUDGE_SCHEMA,
                messages=[
                    {"role": "system", "content": "You are a neutral translation evaluator."},
                    {"role": "user", "content": prompt}
                ]
            )
            return json.loads(response.choices[0].message.content)
        except Exception as e:
            if attempt < max_retries - 1:
                time.sleep(5)
                continue
            return None
    return None

def save_judge_results_jsonl(model_name, results, output_dir):
    """Save LLM-as-a-Judge results in required format."""
    filename = f"{GROUP_NAME}-hw2_transl-judge-{model_name}.jsonl"
    filepath = output_dir / filename

    with open(filepath, "w", encoding="utf-8") as f:
        for result in results:
            json.dump(result, f, ensure_ascii=False)
            f.write("\n")

    print(f"Saved: {filename}")
    return filepath

In [19]:
if RUN_LLM_JUDGE:
    if not OPENAI_API_KEY:
        print("OpenAI API key required for LLM-as-a-Judge. Skipping.")
    else:
        print("Running LLM-as-a-Judge evaluation...\n")
        client = OpenAI(api_key=OPENAI_API_KEY)

        # Collect all model results
        all_model_results = {**finetuned_results, **gpt4_results, **gemini_results}

        if not all_model_results:
            print("No model results to evaluate. Run models first.")
        else:
            judge_results = {}

            for model_name, predictions in all_model_results.items():
                print(f"\nEvaluating: {model_name}")

                evaluated = []

                for pred in tqdm(predictions, desc=f"Judging {model_name}", leave=False):
                    # Skip failed translations
                    if pred["prediction"] == "[FAILED]":
                        continue

                    evaluation = judge_translation(
                        source=pred["source"],
                        prediction=pred["prediction"],
                        reference=None,  # No reference for archaic Italian
                        client=client
                    )

                    if evaluation:
                        evaluated.append({
                            "id": pred["id"],
                            "source": pred["source"],
                            "prediction": pred["prediction"],
                            "scores": {
                                k: evaluation[k]["score"]
                                for k in ["Faithfulness", "Fluency", "Style", "Overall"]
                            },
                            "feedback": {
                                k: evaluation[k]["feedback"]
                                for k in ["Faithfulness", "Fluency", "Style", "Overall"]
                            }
                        })

                    time.sleep(1)  # Rate limiting

                # Calculate summary statistics
                if evaluated:
                    import statistics
                    summary = {}
                    for criterion in ["Faithfulness", "Fluency", "Style", "Overall"]:
                        scores = [r["scores"][criterion] for r in evaluated]
                        summary[criterion] = {
                            "mean": round(statistics.mean(scores), 2),
                            "std": round(statistics.pstdev(scores), 2)
                        }

                    print(f"   Average scores: ", end="")
                    print(" | ".join([f"{k}={v['mean']:.2f}" for k, v in summary.items()]))

                    # Save results
                    save_judge_results_jsonl(model_name, evaluated, OUTPUT_PATH)
                    judge_results[model_name] = evaluated

            print(f"\nLLM-as-a-Judge evaluation complete!")
            print(f"   Evaluated {len(judge_results)} models")
else:
    print("Skipping LLM-as-a-Judge evaluation")
    judge_results = {}

Running LLM-as-a-Judge evaluation...


Evaluating: a1_lexical_expert


Judging a1_lexical_expert:   0%|          | 0/97 [00:00<?, ?it/s]

   Average scores: Faithfulness=3.71 | Fluency=4.27 | Style=4.00 | Overall=3.75
Saved: exACSAI-hw2_transl-judge-a1_lexical_expert.jsonl

Evaluating: a1_syntactic_expert


Judging a1_syntactic_expert:   0%|          | 0/97 [00:00<?, ?it/s]

KeyboardInterrupt: 

## Summary and Results

All output files have been generated in the required `.jsonl` format.

In [20]:
# List all generated files
print("\n" + "="*60)
print("GENERATED OUTPUT FILES")
print("="*60)

output_files = sorted(OUTPUT_PATH.glob("*.jsonl"))

if output_files:
    print(f"\nTotal files: {len(output_files)}\n")

    # Translation outputs
    transl_files = [f for f in output_files if "judge" not in f.name]
    if transl_files:
        print("Translation Outputs:")
        for f in transl_files:
            print(f"{f.name}")

    # Judge outputs
    judge_files = [f for f in output_files if "judge" in f.name]
    if judge_files:
        print("\nLLM-as-a-Judge Outputs:")
        for f in judge_files:
            print(f"{f.name}")
else:
    print("\nNo output files generated. Check configuration settings.")

print("\n" + "="*60)
print("EVALUATION COMPLETE")
print("="*60)
print("\nAll required files are ready for submission!")
print(f"\nDownload from: {OUTPUT_PATH.absolute()}")


GENERATED OUTPUT FILES

Total files: 11

Translation Outputs:
exACSAI-hw2_transl-a1_lexical_expert.jsonl
exACSAI-hw2_transl-a1_merged_full.jsonl
exACSAI-hw2_transl-a1_semantic_expert.jsonl
exACSAI-hw2_transl-a1_syntactic_expert.jsonl
exACSAI-hw2_transl-a2_early_expert.jsonl
exACSAI-hw2_transl-a2_late_expert.jsonl
exACSAI-hw2_transl-a2_merged_full.jsonl
exACSAI-hw2_transl-a2_middle_expert.jsonl
exACSAI-hw2_transl-gpt4.1-zeroshot.jsonl
exACSAI-hw2_transl-unified_merged_full.jsonl

LLM-as-a-Judge Outputs:
exACSAI-hw2_transl-judge-a1_lexical_expert.jsonl

EVALUATION COMPLETE

All required files are ready for submission!

Download from: /workspace/outputs


## Notes

### File Naming Convention

All output files follow the required naming format:
- Translation outputs: `exACSAI-hw2_transl-{model_name}.jsonl`
- Judge outputs: `exACSAI-hw2_transl-judge-{model_name}.jsonl`

### Models Evaluated

**Fine-tuned Experts:**
- Approach 1: Lexical, Syntactic, Semantic experts + Merged
- Approach 2: Early, Middle, Late period experts + Merged
- Unified: Combined merger of both approaches

**LLM Baselines:**
- GPT-4.1 (zero-shot and few-shot)
- Gemini 2.0 (zero-shot and few-shot)

### Evaluation Criteria

LLM-as-a-Judge evaluates on:
1. **Faithfulness**: Meaning preservation (1-5)
2. **Fluency**: Grammar and readability (1-5)
3. **Style**: Naturalness and human-likeness (1-5)
4. **Overall**: Holistic quality (1-5)

---

**For more details, see our full report.**